# 02 — Baseline: DummyClassifier i LogisticRegression w sklearn Pipeline (Sprint 1)

Wynik notebooka: `metrics.json` w katalogu głównym repozytorium (artefakt Definition of Done).

- Model preprocessing (imputacja, skalowanie, kodowanie) żyje wewnątrz sklearn Pipeline
  zbudowanego w `build_baseline_pipeline()` (`src/asi_project/model.py`, TODO zespołu).
- Do X trafiają wyłącznie ALLOWED FEATURES zadeklarowane w `project.yaml`.
- `metrics.json` w repozytorium musi pochodzić z pełnych danych. CI uruchamia ten notebook
  od początku do końca na próbce (`ASI_DATA_MODE=sample`) i zapisuje wynik poza repozytorium.
- Odpowiedzi tekstowe wpisujcie w miejsce `*(odpowiedź)*`.

## 0. Ustawienia

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay

from asi_project.config import get_contract, load_project_config
from asi_project.data import (
    assert_valid,
    filter_training_rows,
    load_dataset,
    make_xy,
    normalize_input,
    split_train_test,
)
from asi_project.metrics import build_metrics_record, compute_metrics, save_metrics
from asi_project.model import build_baseline_pipeline, make_dummy

config = load_project_config()
contract = get_contract(config)
DATA_MODE = os.environ.get("ASI_DATA_MODE", "full")
METRICS_PATH = Path(os.environ.get("ASI_METRICS_PATH", config.metrics_path))

print(f"Dataset: {contract.dataset_id} | target: {contract.target}")
print(f"Klasa pozytywna: {contract.positive_class} | split: {contract.split_strategy}")
print(f"Tryb danych: {DATA_MODE} | wynik: {METRICS_PATH}")

## 1. Dane: input normalization, kontrakt, training-only filtering

Kolejność jest celowa: najpierw input normalization (bezpieczna przy predykcji), potem
sprawdzenie kontraktu, na końcu training-only filtering (tylko dla danych treningowych).

In [ ]:
df_raw = load_dataset(config, contract, mode=DATA_MODE)
df = normalize_input(df_raw)
assert_valid(df, contract, config)
df = filter_training_rows(df, contract)
print(f"Rekordy: {len(df_raw)} -> po training-only filtering: {len(df)}")

## 2. X i y

`make_xy` buduje X wyłącznie z zadeklarowanych ALLOWED FEATURES i koduje target binarnie:
klasa pozytywna = 1.

In [ ]:
X, y = make_xy(df, contract, config.model_features)
print(f"X: {X.shape} | cechy: {list(X.columns)}")
print(f"Udział klasy pozytywnej (y = 1): {y.mean():.3f}")

## 3. Split train/test

Strategia splitu pochodzi z kontraktu datasetu (`stratified_random` albo `time_ordered`).
Split wykonujemy **przed** jakimkolwiek `fit`.

In [ ]:
X_train, X_test, y_train, y_test = split_train_test(
    X,
    y,
    contract,
    test_size=config.test_size,
    random_state=config.random_state,
)
print(f"train: {len(X_train)} rekordów, udział klasy pozytywnej {y_train.mean():.3f}")
print(f"test:  {len(X_test)} rekordów, udział klasy pozytywnej {y_test.mean():.3f}")

## 4. DummyClassifier

In [ ]:
dummy = make_dummy()
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)

## 5. sklearn Pipeline: model preprocessing + LogisticRegression

Pipeline buduje funkcja `build_baseline_pipeline()` w `src/asi_project/model.py`
(**TODO zespołu**). Wymagania opisuje docstring funkcji; sprawdzają je testy w
`tests/test_project.py`.

In [ ]:
pipeline = build_baseline_pipeline(
    list(config.numeric_features),
    list(config.categorical_features),
)
pipeline.fit(X_train, y_train)
lr_pred = pipeline.predict(X_test)
pipeline

## 6. `predict_proba`

In [ ]:
proba = pipeline.predict_proba(X_test)
print("Kolejność klas w predict_proba:", pipeline.classes_.tolist())

# TODO (S1): wybierz z `proba` kolumnę z prawdopodobieństwem klasy pozytywnej (y = 1).
# Wskazówka: kolejność kolumn opisuje pipeline.classes_.
positive_proba = None

assert proba.shape == (len(X_test), 2)
assert np.allclose(proba.sum(axis=1), 1.0)
print(f"Średnie prawdopodobieństwo klasy pozytywnej: {np.mean(positive_proba):.3f}")

## 7. Metryki (klasa pozytywna = 1)

In [ ]:
results = {
    "dummy": compute_metrics(y_test, dummy_pred),
    "logistic_regression": compute_metrics(y_test, lr_pred),
}
pd.DataFrame(results).T.drop(columns="confusion_matrix")

## 8. Confusion matrix

In [ ]:
_, ax = plt.subplots(figsize=(4, 4))
ConfusionMatrixDisplay.from_predictions(y_test, lr_pred, labels=[0, 1], ax=ax)
ax.set_title("LogisticRegression (1 = klasa pozytywna)")
plt.show()

## 9. Zapis `metrics.json`

`save_metrics` sprawdza kontrakt pliku przed zapisem. W repozytorium commitujemy wynik
z pełnych danych (`ASI_DATA_MODE` nieustawione).

In [ ]:
record = build_metrics_record(
    contract=contract,
    config=config,
    features=list(pipeline.feature_names_in_),
    n_train=len(X_train),
    n_test=len(X_test),
    data_mode=DATA_MODE,
    results=results,
)
save_metrics(record, METRICS_PATH, contract=contract)
print(f"Zapisano: {METRICS_PATH}")

## 10. Pytania interpretacyjne (1–3 zdania każde)

1. Dlaczego kontrakt waszego datasetu wymaga tej strategii splitu? Co stałoby się z oceną
   modelu przy innej strategii?

*(odpowiedź)*

2. Czy LogisticRegression jest lepsza od DummyClassifier na waszej metryce głównej? Podajcie
   liczby z tabeli metryk.

*(odpowiedź)*

3. Co w waszym problemie oznacza błąd typu FP, a co FN (perspektywa użytkownika systemu)?

*(odpowiedź)*

4. Gdzie w tym notebooku działa model preprocessing i dlaczego nie ma go w `normalize_input()`?

*(odpowiedź)*